<a href="https://colab.research.google.com/github/pranju20/Autonomous-Code-Auditor-Agent-/blob/main/CodeAuditorGenAIProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain-huggingface langgraph langchain

In [2]:
from typing import TypedDict, List

class AgentState(TypedDict):
    code: str
    error_found: bool
    feedback: str
    iterations: int

In [3]:
import os
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
model_id = "Qwen/Qwen2.5-7B-Instruct"

llm = HuggingFaceEndpoint(
    repo_id=model_id,
    task="text-generation",
    temperature=0.1,)

chat_model = ChatHuggingFace(llm=llm)

In [4]:
def auditor_node(state: AgentState):
    print("---AUDITING CODE---")
    code = state["code"]

    prompt = f"Does this Python code have an error? Answer ONLY 'YES' or 'NO'. Code: {code}"
    response = chat_model.invoke(prompt)

    found = "YES" in response.content.upper()
    return {"error_found": found, "feedback": "Logic error detected", "iterations": state["iterations"] + 1}

def fixer_node(state: AgentState):
    print("---FIXING CODE---")
    prompt = f"Fix the error in this code and return ONLY the corrected code: {state['code']}"
    response = chat_model.invoke(prompt)

    return {"code": response.content, "error_found": False}

In [5]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)

workflow.add_node("auditor", auditor_node)
workflow.add_node("fixer", fixer_node)

workflow.set_entry_point("auditor")

def router(state):
    if state["error_found"] and state["iterations"] < 3:
        return "fixer"
    else:
        return END

workflow.add_conditional_edges("auditor", router)

workflow.add_edge("fixer", "auditor")

app = workflow.compile()

In [6]:
initial_code = "def hello_world() print('hello')"

inputs = {"code": initial_code, "iterations": 0, "error_found": False, "feedback": ""}
for output in app.stream(inputs):
    print(output)

---AUDITING CODE---
{'auditor': {'error_found': True, 'feedback': 'Logic error detected', 'iterations': 1}}
---FIXING CODE---
{'fixer': {'code': "```python\ndef hello_world():\n    print('hello')\n```", 'error_found': False}}
---AUDITING CODE---
{'auditor': {'error_found': False, 'feedback': 'Logic error detected', 'iterations': 2}}
